# Cell-Akamai companion 02 · Units and memory budget
**Local Ollama on this host — NOT an Akamai GPU run, NOT vLLM.** Original workshop notebook `02_units_and_memory_budget/02_units_and_memory_budget.ipynb` is unchanged; this notebook consumes the same *question* on a different backend. Results are never pooled with Akamai runs. Status: **NOT EXECUTED** until the executed copy exists.

Mostly **modeled** (arithmetic from model metadata). The one **measured** quantity is `ollama ps` memory for a loaded model. KV bytes/token is exact only for standard attention; hybrid layouts (e.g. LFM2) are `NOT_COMPUTED`, never guessed. KV dtype f16 is an **assumption**.

In [ ]:
import sys, json, pathlib
sys.path[:0] = ["lane_b/companion", "lane_b/harness"]   # run from the repo root
import cakm, pandas as pd
OUT = pathlib.Path("runs/companion"); OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["hf.co/LiquidAI/LFM2.5-350M-GGUF:Q4_K_M", "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M"]  # small models; edit to taste
HOST = cakm.host_info(); print(json.dumps(HOST, indent=1))

In [ ]:
import requests
MODELS2 = MODELS + ["qwen3:1.7b"]  # Qwen3 family for continuity with Akamai's Qwen3-4B; standard attention => exact KV math
GiB = 2**30; PTR_BYTES = 71; PTR_EVERY = 512   # custody pointer = "sha256:"+64 hex chars; one per 512 tokens (PARAMETER, edit)
rows = []
for m in MODELS2:
    meta = cakm.model_meta(m); mi = meta["model_info"]; params = mi.get("general.parameter_count")
    kv = cakm.kv_bytes_per_token(mi)
    rows.append({"model": m, "digest": meta["digest"], "params": params, "quant": meta["quantization"], "file_bytes": meta["file_bytes"],
                 "bytes_per_param_effective": meta["file_bytes"] / params if params else None,
                 "kv_bytes_per_token_f16": kv, "kv_status": "MODELED_EXACT_STANDARD_ATTENTION" if kv else "NOT_COMPUTED_HYBRID_OR_UNKNOWN_LAYOUT",
                 "tokens_per_GiB_kv": (GiB / kv) if kv else None,
                 "kv_MiB_at_4096": (kv * 4096 / 2**20) if kv else None,
                 "custody_ptr_bytes_per_1k_tokens": PTR_BYTES * 1000 / PTR_EVERY,
                 "custody_ptr_vs_kv_ratio": (PTR_BYTES / PTR_EVERY) / kv if kv else None,
                 "context_length_max": mi.get(f"{mi['general.architecture']}.context_length")})
df = pd.DataFrame(rows); df

In [ ]:
# MEASURED: load each model at num_ctx=4096 (one-token generation), read memory from /api/ps. Compare with modeled weights + KV.
meas = []
for m in MODELS2:
    cakm.generate_stream(m, "Hello", num_predict=1, options={"num_ctx": 4096})
    ps = [x for x in requests.get(cakm.OLLAMA + "/api/ps").json()["models"] if x["name"] == m]
    ps = ps[0] if ps else {}
    r = next(x for x in rows if x["model"] == m)
    modeled = r["file_bytes"] + (r["kv_bytes_per_token_f16"] * 4096 if r["kv_bytes_per_token_f16"] else 0)
    meas.append({"model": m, "ps_size_bytes": ps.get("size"), "ps_size_vram_bytes": ps.get("size_vram"),
                 "modeled_weights_plus_kv_bytes": modeled if r["kv_bytes_per_token_f16"] else None,
                 "ratio_measured_over_modeled": (ps.get("size") / modeled) if ps.get("size") and r["kv_bytes_per_token_f16"] else None})
    requests.post(cakm.OLLAMA + "/api/generate", json={"model": m, "keep_alive": 0})  # unload
mdf = pd.DataFrame(meas); mdf

In [ ]:
rec = cakm.write_receipt(OUT/"02", "memory_budget", rows + [{"measured": x} for x in meas], {"module": "02", "host": HOST,
        "assumptions": {"kv_dtype": "f16 (assumed Ollama default, not verified)", "ptr_every_tokens": PTR_EVERY, "ptr_bytes": PTR_BYTES}})
rec["jsonl_sha256"]

**Reading it:** a measured/modeled ratio far from 1 is reported as observed, not explained — Ollama may allocate compute buffers and use a different KV layout than this arithmetic. Unified memory is shared with the OS, so this is not an Akamai VRAM comparison.